In [1]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import awkward as ak
import ROOT
import vector
vector.register_awkward()

In [2]:
def print_cutflow(arrays, label):
    n_events = len(arrays["trk_pt"])
    n_tracks = ak.sum(ak.num(arrays["trk_pt"]))
    print(f"{label:50s}  events: {n_events:>8,}  tracks: {n_tracks:>8,}")


In [3]:
def dR_mask(arrays, obj_prefix, trk_prefix="trk", dr_cut=0.5, pt_cut=None, mode="all"):
    tracks = ak.zip({
        "eta": arrays[f"{trk_prefix}_eta"],
        "phi": arrays[f"{trk_prefix}_phi"]
    })
    objs = ak.zip({
        "eta": arrays[f"{obj_prefix}_eta"],
        "phi": arrays[f"{obj_prefix}_phi"],
        "pt":  arrays[f"{obj_prefix}_pt"],
    })
    
    if pt_cut is not None:
        objs = objs[objs.pt > pt_cut]
    
    t, o  = ak.unzip(ak.cartesian([tracks, objs], nested=True))
    dphi  = np.arctan2(np.sin(t.phi - o.phi), np.cos(t.phi - o.phi))
    dR    = np.sqrt((t.eta - o.eta)**2 + dphi**2)
    
    if mode == "all":
        return ak.all(dR > dr_cut, axis=2)  # far from ALL objects
    elif mode == "any":
        return ak.any(dR > dr_cut, axis=2)  # far from AT LEAST ONE object

In [4]:
def print_cutflow(object_cuts, event_cuts=None):
    """
    Print individual and cumulative cutflow tables for arbitrary selections.

    Parameters
    ----------
    object_cuts : list of dict
        Each dict defines one group of per-object cuts applied sequentially.
        Keys:
            "label"  : str  — group heading printed in the table
            "cuts"   : dict[str, ak.Array]  — ordered {cut_name: boolean_mask}
                       masks must be per-object (shape: events x objects)
        Groups are applied in order. An event passes a group if at least one
        object survives all cuts in that group. The surviving-object mask from
        each group is ANDed with the next group's candidates (cross-object
        requirements are not supported; use event_cuts for those).

    event_cuts : dict[str, ak.Array], optional
        Per-event boolean masks (shape: events) applied after all object_cuts.
        These are cumulative with each other and with the object selection.

    Returns
    -------
    int
        Number of events passing the full selection.

    Examples
    --------
    muon_cuts = {
        "muon: isTrigMatched": arrays["muon_isTrigMatched"],
        "muon: pt > 26":       arrays["muon_pt"] > 26,
    }
    track_cuts = {
        "trk: pt > 30":        arrays["trk_pt"] > 30,
        "trk: |eta| < 2.1":   abs(arrays["trk_eta"]) < 2.1,
    }
    n = print_cutflow(
        object_cuts=[
            {"label": "Muon cuts",  "cuts": muon_cuts},
            {"label": "Track cuts", "cuts": track_cuts},
        ],
        event_cuts={"MET > 100": met > 100},
    )
    """

    # Infer total event count from the first mask in the first group
    first_mask = next(iter(object_cuts[0]["cuts"].values()))
    n_total = len(first_mask)

    col = 50  # label column width

    def _header(title):
        print(f"\n{'=' * (col + 24)}")
        print(f"{title:^{col + 24}}")
        print(f"{'=' * (col + 24)}")

    def _divider():
        print(f"{'-' * (col + 24)}")

    def _row(label, n, denom, indent=2):
        pct = f"{n / denom:.2%}" if denom else "N/A"
        print(f"{' ' * indent}{label:<{col - indent}} {n:>8,}  {pct:>10}")

    # ── Individual yields ────────────────────────────────────────────────────
    _header("INDIVIDUAL CUT YIELDS")
    print(f"{'Cut':<{col}} {'Passing':>8}  {'Efficiency':>10}")
    _divider()

    for group in object_cuts:
        print(f"  -- {group['label']} --")
        for name, mask in group["cuts"].items():
            n = ak.sum(ak.any(mask, axis=1))
            _row(name, n, n_total)

    if event_cuts:
        print(f"  -- Event cuts --")
        for name, mask in event_cuts.items():
            _row(name, ak.sum(mask), n_total)

    # ── Cumulative cutflow ───────────────────────────────────────────────────
    _header("CUMULATIVE CUTFLOW")
    print(f"{'Cut':<{col}} {'Passing':>8}  {'Cumul. Eff.':>10}")
    _divider()
    print(f"{'Total events':<{col}} {n_total:>8,}  {'100.00%':>10}")

    cumul_event_mask = ak.Array([True] * n_total)

    for group in object_cuts:
        print(f"  -- {group['label']} --")
        cumul_obj_mask = None
        for name, mask in group["cuts"].items():
            cumul_obj_mask = mask if cumul_obj_mask is None else cumul_obj_mask & mask
            n = ak.sum(cumul_event_mask & ak.any(cumul_obj_mask, axis=1))
            _row(name, n, n_total)
        cumul_event_mask = cumul_event_mask & ak.any(cumul_obj_mask, axis=1)

    if event_cuts:
        print(f"  -- Event cuts --")
        for name, mask in event_cuts.items():
            cumul_event_mask = cumul_event_mask & mask
            _row(name, ak.sum(cumul_event_mask), n_total)

    _divider()
    final = int(ak.sum(cumul_event_mask))
    print(f"{'Final selection':<{col}} {final:>8,}  {final / n_total:>10.2%}")
    print(f"{'=' * (col + 24)}\n")

    return final

In [6]:
NTUPLE_FILE = "root://cmseosmgm01.fnal.gov:1094//store/user/delossan/ntuple.root"
TREE_NAME   = "ntuplizer/Events" 
f    = uproot.open(NTUPLE_FILE)
tree = f[TREE_NAME]
print(f"Entries: {tree.num_entries:,}\n")
print("Branches:")
for b in tree.keys():
    print(f"  {b}")

arrays = tree.arrays(["muon_pt", "muon_eta", "muon_phi", "muon_charge", "muon_isTight", 
                      "muon_pfRelIso04_dBeta", "trk_hp_numberOfValidPixelHits", "trk_hp_numberOfValidHits",
                      "ele_pt", "muon_isTrigMatched", "trk_missingInnerHits", "trk_hitDrop_missingMiddleHits",
                      "ele_eta", "ele_phi", "tau_pt", "tau_eta", "tau_phi", "trk_pt", "trk_dz", 
                      "trk_eta","trk_hp_trackerLayersWithMeasurement", "trk_hp_trackerLayersWithoutMeasurement_OUTER",
                      "trk_hp_trackerLayersWithMeasurement","trk_relativePFIso", "trk_phi", "trk_eta", "trk_caloTotal",
                      "trk_charge", "trk_minDRToMaskedEcal", "trk_theta", "trk_relativePFIso", "trk_dxy",
                      "jet_pt", "jet_phi", "jet_eta", "jet_isTightLepVeto"],
                     entry_stop=5_000_000, library="ak")
# Boolean per event: does it have at least one muon with pt > 26
has_high_pt_muon = ak.any(arrays["muon_pt"] > 26, axis=1)

print(f"Events with at least one muon (pt > 26): {ak.sum(has_high_pt_muon):,}")

Entries: 499

Branches:
  run
  lumi
  eventNum
  met_pt
  met_phi
  metNoMu_pt
  metNoMu_phi
  rho_all
  rho_allCalo
  rho_centralCalo
  trk_pt
  trk_eta
  trk_theta
  trk_phi
  trk_dxy
  trk_dxyError
  trk_dz
  trk_dzError
  trk_charge
  trk_fromPV
  trk_deltaEta
  trk_deltaPhi
  trk_minDRToMaskedEcal
  trk_isHighPurityTrack
  trk_isTightTrack
  trk_isLooseTrack
  trk_dEdxStrip
  trk_dEdxPixel
  trk_caloEm
  trk_caloHad
  trk_caloTotal
  trk_caloTotNoPU
  trk_crossedEcalStatus
  trk_crossedHcalStatus
  trk_pfIso
  trk_relativePFIso
  trk_pfIso_chHad
  trk_pfIso_neutHad
  trk_pfIso_photon
  trk_pfIso_puChHad
  trk_miniIso_chHad
  trk_miniIso_neutHad
  trk_miniIso_photon
  trk_miniIso_puChHad
  trk_miniIso_relative
  trk_missingInnerHits
  trk_missingMiddleHits
  trk_missingOuterHits
  trk_hitDrop_missingMiddleHits
  trk_dPhiMet
  trk_dPhiMetNoMu
  trk_ptOverMetNoMu
  trk_hp_numberOfAllHits
  trk_hp_numberOfAllTrackerHits
  trk_hp_numberOfValidHits
  trk_hp_numberOfValidTrackerHits
  t

In [7]:
import numpy as np


def delta_r(eta1, phi1, eta2, phi2):
    deta = eta1 - eta2
    dphi = phi1 - phi2
    # Wrap dphi into [-pi, pi]
    dphi = ak.where(dphi >  np.pi, dphi - 2*np.pi, dphi)
    dphi = ak.where(dphi < -np.pi, dphi + 2*np.pi, dphi)
    return np.sqrt(deta**2 + dphi**2)

In [8]:
ak.sum(arrays["jet_isTightLepVeto"] == 0)

np.int64(452)

In [9]:
jet_cuts = (
    (arrays["jet_pt"] > 30)
    & (abs(arrays["jet_eta"]) < 4.5)
    & (arrays["jet_isTightLepVeto"])   # arrays(...) was a function call — should be array indexing with []
)
trk_arr = ak.zip({"trk_eta": arrays["trk_eta"], "trk_phi": arrays["trk_phi"]})
jet_arr = ak.zip({"jet_eta": arrays["jet_eta"][jet_cuts], "jet_phi": arrays["jet_phi"][jet_cuts]})

trk_jet_pairs = ak.cartesian(
    {"trk": trk_arr, "jet": jet_arr},
    nested=True)

dRMinJet = ak.fill_none(
    ak.min(
        delta_r(
            trk_jet_pairs['trk']['trk_eta'],
            trk_jet_pairs['trk']['trk_phi'],
            trk_jet_pairs['jet']['jet_eta'],
            trk_jet_pairs['jet']['jet_phi']),
            axis=2,

    ),
    999.0
)

In [10]:
good_muon = (
    (arrays["muon_isTrigMatched"])
    & (arrays["muon_pt"] > 26)
    & (abs(arrays["muon_eta"]) < 2.1)
    & (arrays["muon_isTight"])
    & (arrays["muon_pfRelIso04_dBeta"] < 0.15)
)
# pi/2
M_PI_2 = 1.57079632679489661923

inTOBCrack = (
    (abs(arrays["trk_dz"]) < 0.5)
    & (abs(M_PI_2 - arrays["trk_theta"]) < 1.0e-3)
)

good_track = (
    (arrays["trk_pt"] > 30)
    & (abs(arrays["trk_eta"]) < 2.1)
    & (
        (abs(arrays["trk_eta"]) < 1.42)
        | (abs(arrays["trk_eta"]) > 1.65)
    )
    & (
        (abs(arrays["trk_eta"]) < 0.15)
        | (abs(arrays["trk_eta"]) > 0.35)
    )
    & (
        (abs(arrays["trk_eta"]) < 1.55)
        | (abs(arrays["trk_eta"]) > 1.85)
    )
    &  ~inTOBCrack
    & (arrays["trk_minDRToMaskedEcal"] > 0.05)
    & (arrays["trk_hp_numberOfValidPixelHits"] >= 4)
    & (arrays["trk_hp_numberOfValidHits"] >= 4)
    & (arrays["trk_missingInnerHits"] == 0)
    & (arrays["trk_hitDrop_missingMiddleHits"] == 0)
    & (arrays["trk_relativePFIso"] < 0.05)
    & (abs(arrays["trk_dxy"]) < 0.02)
    & (abs(arrays["trk_dz"]) < 0.5)
    & (abs(dRMinJet) > 0.5)
    
        
)

has_good_muon = ak.any(good_muon, axis=1)
has_good_track = ak.any(good_track, axis=1)

event_mask = has_good_muon & has_good_track



n_passing = ak.sum(event_mask)

print(f"Events passing: {n_passing:,}")
# 
#print(f"Events with at least one muon (pt > 26): {ak.sum(has_high_pt_muon):,}")

Events passing: 103


In [13]:
M_PI_2 = 1.57079632679489661923
inTOBCrack = (
    (abs(arrays["trk_dz"]) < 0.5)
    & (abs(M_PI_2 - arrays["trk_theta"]) < 1.0e-3)
)

n_passing = print_cutflow(
    object_cuts=[
        {
            "label": "Muon cuts",
            "cuts": {
                "muon: isTrigMatched":               arrays["muon_isTrigMatched"],
                "muon: pt > 26":                     arrays["muon_pt"] > 26,
                "muon: |eta| < 2.1":                abs(arrays["muon_eta"]) < 2.1,
                "muon: isTight":                     arrays["muon_isTight"],
                "muon: pfRelIso04_dBeta < 0.15":     arrays["muon_pfRelIso04_dBeta"] < 0.15,
            },
        },
        {
            "label": "Track cuts",
            "cuts": {
                "trk: pt > 30":                          arrays["trk_pt"] > 30,
                "trk: |eta| < 2.1":                     abs(arrays["trk_eta"]) < 2.1,
                "trk: not in ECAL crack (eta 1.42-1.65)": (abs(arrays["trk_eta"]) < 1.42) | (abs(arrays["trk_eta"]) > 1.65),
                "trk: not in ECAL crack (eta 0.15-0.35)": (abs(arrays["trk_eta"]) < 0.15) | (abs(arrays["trk_eta"]) > 0.35),
                "trk: not in ECAL crack (eta 1.55-1.85)": (abs(arrays["trk_eta"]) < 1.55) | (abs(arrays["trk_eta"]) > 1.85),
                "trk: not in TOB crack":                  ~inTOBCrack,
                "trk: minDRToMaskedEcal > 0.05":          arrays["trk_minDRToMaskedEcal"] > 0.05,
                "trk: nValidPixelHits >= 4":              arrays["trk_hp_numberOfValidPixelHits"] >= 4,
                "trk: nValidHits >= 4":                   arrays["trk_hp_numberOfValidHits"] >= 4,
                "trk: missingInnerHits == 0":             arrays["trk_missingInnerHits"] == 0,
                "trk: hitDrop_missingMiddleHits == 0":    arrays["trk_hitDrop_missingMiddleHits"] == 0,
                "trk: relativePFIso < 0.05":              arrays["trk_relativePFIso"] < 0.05,
                "trk: |dxy| < 0.02":                      abs(arrays["trk_dxy"]) < 0.02,
                "trk: |dz| < 0.5":                        abs(arrays["trk_dz"]) < 0.5,
                "trk: dR(jet) > 0.5":                     abs(dRMinJet) > 0.5,
            },
        },
    ],
)

assert n_passing == ak.sum(has_good_muon & has_good_track)  # sanity check


                          INDIVIDUAL CUT YIELDS                           
Cut                                                 Passing  Efficiency
--------------------------------------------------------------------------
  -- Muon cuts --
  muon: isTrigMatched                                   498      99.80%
  muon: pt > 26                                         444      88.98%
  muon: |eta| < 2.1                                     498      99.80%
  muon: isTight                                         474      94.99%
  muon: pfRelIso04_dBeta < 0.15                         489      98.00%
  -- Track cuts --
  trk: pt > 30                                          383      76.75%
  trk: |eta| < 2.1                                      494      99.00%
  trk: not in ECAL crack (eta 1.42-1.65)                491      98.40%
  trk: not in ECAL crack (eta 0.15-0.35)                492      98.60%
  trk: not in ECAL crack (eta 1.55-1.85)                490      98.20%
  trk: not in TOB cr

In [ ]:
# Calculate Fiducial Map 
ak.sum(dRMinJet < 0.5)
dRMinJet

In [ ]:
print(ak.type(dRMinJet))
print(len(dRMinJet))

In [ ]:
print(ak.sum(jet_cuts, axis=1)[:20])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dRMinJet_flat = ak.to_numpy(ak.flatten(dRMinJet))

# Exclude the fill_none sentinel value (999.0) for plotting
dRMinJet_flat = dRMinJet_flat[dRMinJet_flat < 999.0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dRMinJet_flat, bins=50, range=(0, 6), histtype="step", linewidth=1.5, color="steelblue")

ax.axvline(0.5, color="red", linestyle="--", linewidth=1.5, label="cut: dRMinJet > 0.5")
ax.set_xlabel("dRMinJet", fontsize=13)
ax.set_ylabel("Tracks", fontsize=13)
ax.set_title("Min ΔR(track, jet)", fontsize=14)
ax.legend()
ax.set_yscale("log")

plt.tight_layout()
plt.savefig("dRMinJet.png", dpi=150)
plt.show()